In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import torch
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
data = pd.read_csv('/content/drive/MyDrive/Data/SentimentTwitterDataset.csv', sep='\t', engine='python')
df = pd.DataFrame(data)

# Map sentiment labels -1, 0, 1 to 0, 1, 2
sentiment_mapping = {-1: 0, 0: 1, 1: 2}
df['sentimen'] = df['sentimen'].map(sentiment_mapping)

# Tokenisasi
tokenizer = Tokenizer(num_words=1000, oov_token="<OOV>")
tokenizer.fit_on_texts(df['Tweet'])
sequences = tokenizer.texts_to_sequences(df['Tweet'])
X = pad_sequences(sequences, maxlen=10, padding='post')

y = df['sentimen'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

class SentimentDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_data = SentimentDataset(X_train, y_train)
test_data = SentimentDataset(X_test, y_test)

train_loader = DataLoader(train_data, batch_size=2, shuffle=True)
test_loader = DataLoader(test_data, batch_size=2)

import torch.nn as nn
import torch.nn.functional as F

class SentimentModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(SentimentModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 3)

    def forward(self, x):
        x = self.embedding(x)
        x = x.mean(dim=1)  # GlobalAveragePooling
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

vocab_size = 1000
embed_dim = 16
hidden_dim = 16

model = SentimentModel(vocab_size, embed_dim, hidden_dim)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


for epoch in range(20):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch).squeeze()
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")
print(outputs)


Mounted at /content/drive
Epoch 1, Loss: 1.2185
Epoch 2, Loss: 0.9569
Epoch 3, Loss: 0.3585
Epoch 4, Loss: 1.5122
Epoch 5, Loss: 0.2487
Epoch 6, Loss: 1.5387
Epoch 7, Loss: 1.0281
Epoch 8, Loss: 1.4231
Epoch 9, Loss: 1.9337
Epoch 10, Loss: 0.3458
Epoch 11, Loss: 1.1896
Epoch 12, Loss: 1.4841
Epoch 13, Loss: 1.1818
Epoch 14, Loss: 0.3357
Epoch 15, Loss: 0.4813
Epoch 16, Loss: 1.1598
Epoch 17, Loss: 0.5410
Epoch 18, Loss: 0.7762
Epoch 19, Loss: 2.5448
Epoch 20, Loss: 0.1929
tensor([[-1.5693,  1.8469, -0.8720],
        [ 1.0851, -0.1150, -2.2013]], grad_fn=<SqueezeBackward0>)
